In [135]:
#Limpiando dataset y exportandolo a CSV
import pandas as pd 

def clean_text(text):
    if not isinstance(text, str):
        return text

    # Normaliza unicode y elimina caracteres raros
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", errors="ignore").decode("utf-8")

    return text


df.columns = [clean_text(c) for c in df.columns]

# Normalizar formato para ML
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)


# Limpieza de valores nulos (seguridad)
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.median()))

#Función para detectar outliers (IQR)

def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper)

outliers_ingreso = detect_outliers_iqr(df["ingreso_mensual"])
outliers_limite = detect_outliers_iqr(df["limite_credito"])
outliers_utilizacion = detect_outliers_iqr(df["porcentaje_utilizacion"])

total = len(df)

print("Outliers ingreso mensual:",
      outliers_ingreso.sum(),
      f"({outliers_ingreso.sum()/total*100:.2f}%)")

print("Outliers límite de crédito:",
      outliers_limite.sum(),
      f"({outliers_limite.sum()/total*100:.2f}%)")

print("Outliers porcentaje de utilización:",
      outliers_utilizacion.sum(),
      f"({outliers_utilizacion.sum()/total*100:.2f}%)")

print("Columnas con caracteres corruptos:",
      [c for c in df.columns if "Ã" in c or "Â" in c])

df.to_csv("novacredit_clientes_limpio.csv", index=False)

print("Dataset limpio, validado y listo para análisis/modelado")

# Distribución de la variable objetivo
df["default_12m"].value_counts()


# Proporción de clases
df["default_12m"].value_counts(normalize=True)


# Verificar que no existan columnas no determinísticas
df.columns



Outliers ingreso mensual: 536 (5.36%)
Outliers límite de crédito: 476 (4.76%)
Outliers porcentaje de utilización: 52 (0.52%)
Columnas con caracteres corruptos: []
Dataset limpio, validado y listo para análisis/modelado


Index(['edad', 'ingreso_mensual', 'antiguedad_cliente_meses',
       'segmento_cliente', 'region', 'canal_adquisicion', 'limite_credito',
       'saldo_promedio', 'porcentaje_utilizacion', 'num_transacciones_mes',
       'pagos_atrasados_12m', 'reclamos_12m', 'tipo_producto', 'tipo_contrato',
       'metodo_pago', 'default_12m'],
      dtype='str')

In [136]:

X = df.drop(columns=["default_12m"])
y = df["default_12m"]

In [137]:
X.head()


,edad,ingreso_mensual,antiguedad_cliente_meses,segmento_cliente,region,canal_adquisicion,limite_credito,saldo_promedio,porcentaje_utilizacion,num_transacciones_mes,pagos_atrasados_12m,reclamos_12m,tipo_producto,tipo_contrato,metodo_pago
0,45,13487.36,1,Básico,Centro,Web,36554.88,13130.88,0.36,20,3,0,Tarjeta,Anual,Efectivo
1,38,11122.22,102,Premium,Norte,Sucursal,44606.35,11564.16,0.26,15,0,0,Tarjeta,Mensual,Transferencia
2,47,9335.42,38,Plus,Centro,Referido,24092.66,3835.81,0.16,16,1,1,Tarjeta,Mensual,Débito automático
3,58,14274.80,48,Básico,Sur,App,39951.95,2512.86,0.06,22,0,1,Tarjeta,Mensual,Efectivo
4,37,27400.24,28,Premium,Centro,Referido,108625.12,12242.81,0.11,23,2,1,Tarjeta,Anual,Efectivo


In [138]:
y.head()

0    1
1    0
2    1
3    0
4    1
Name: default_12m, dtype: int64

El dataset fue validado verificando la consistencia estructural, la ausencia de valores nulos y duplicados, así como la correcta definición de los tipos de datos.
Todas las variables predictoras son numéricas, lo que lo hace adecuado para la aplicación de **técnicas de modelación supervisada**.  
La variable objetivo **`default_12m`** es **binaria** y fue separada explícitamente de las variables predictoras, garantizando un proceso de entrenamiento reproducible y controlado.


In [139]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

X = X.select_dtypes(include=["int64", "float64", "bool"])


In [140]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((8000, 9), (2000, 9))

In [141]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
grid_rf.best_params_

In [ ]:
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt


In [ ]:
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC Random Forest:", roc_auc_score(y_test, y_prob))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Default", "Default"]
)

disp.plot(cmap="Blues")
plt.title("Matriz de Confusión – Random Forest")
plt.show()



El modelo de Random Forest demuestra una alta capacidad para identificar clientes con riesgo de *default*, logrando detectar aproximadamente el **87 %** de los casos reales de incumplimiento, con una **precisión del 92 %** en sus predicciones de riesgo.

Esto permite reducir significativamente el otorgamiento de crédito a clientes de alto riesgo, manteniendo al mismo tiempo un **nivel aceptable de rechazo de clientes solventes**.  

El desempeño global del modelo, respaldado por un **AUC–ROC de 0.89**, confirma que es adecuado como herramienta de apoyo a la toma de decisiones en procesos de evaluación crediticia.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

gb = GradientBoostingClassifier(
    random_state=42
)

param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0]
}

grid_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_
grid_gb.best_params_


In [ ]:
y_pred_gb = best_gb.predict(X_test)
y_prob_gb = best_gb.predict_proba(X_test)[:, 1]


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report

# Predicciones
y_pred_gb = best_gb.predict(X_test)
y_prob_gb = best_gb.predict_proba(X_test)[:, 1]

# Métricas
accuracy_gb = accuracy_score(y_test, y_pred_gb)
precision_gb = precision_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)
roc_auc_gb = roc_auc_score(y_test, y_prob_gb)

print("Accuracy GB:", round(accuracy_gb, 4))
print("Precision GB:", round(precision_gb, 4))
print("Recall GB:", round(recall_gb, 4))
print("AUC-ROC GB:", round(roc_auc_gb, 4))

print(classification_report(y_test, y_pred_gb))


### Evaluación y diagnóstico del modelo Gradient Boosting *precision*, *recall* y *AUC‑ROC*.

En términos de detección del riesgo, el modelo alcanza un **recall de 0.92 para la clase Default**, lo que significa que identifica correctamente el **91.7 % de los clientes que efectivamente incurren en incumplimiento**. Este resultado es especialmente relevante en un contexto de evaluación crediticia, ya que reduce significativamente la probabilidad de aprobar créditos a clientes de alto riesgo. Adicionalmente, la **precision de 0.89** para esta misma clase indica que la mayoría de los clientes clasificados como riesgosos realmente lo son, limitando el número de falsos positivos.

El **AUC‑ROC de 0.895** confirma una **alta capacidad discriminativa** del modelo, evidenciando que Gradient Boosting separa de manera efectiva a los clientes con y sin riesgo de default en una amplia gama de umbrales de decisión.

El análisis de la **matriz de confusión** refuerza estas conclusiones. De **1,517 clientes que realmente incurrieron en default**, el modelo identificó correctamente **1,391 casos**, mostrando una clara priorización de la detección del riesgo. Por otro lado, se registraron **175 falsos positivos**, correspondientes a clientes solventes clasificados como riesgosos, un comportamiento aceptable dentro de una estrategia conservadora de gestión del riesgo. Asimismo, se observaron **126 falsos negativos**, que representan clientes que incurrieron en default y no fueron detectados, constituyendo un área potencial de mejora si el objetivo del negocio es minimizar al máximo la exposición al riesgo financiero.





# Confusion Matrix — Gradient Boosting
cm_gb = confusion_matrix(y_test, y_pred_gb)
disp_gb = ConfusionMatrixDisplay(
    confusion_matrix=cm_gb,
    display_labels=["No Default", "Default"]
)
disp_gb.plot(cmap="Blues")
plt.title("Matriz de Confusión – Gradient Boosting")
plt.show()







### Evaluación y diagnóstico del modelo Gradient Boostingplimiento.

Por otro lado, se identificaron **126 falsos negativos**, es decir, clientes que incurrieron en default pero no fueron detectados por el modelo. Aunque este número es relativamente bajo en comparación con los verdaderos positivos, representa un área potencial de mejora si el objetivo del negocio es minimizar al máximo el riesgo financiero.

En conjunto, los resultados indican que el modelo **Gradient Boosting prioriza la detección del default**, ofreciendo un balance adecuado entre control de riesgo y aceptación de clientes. Esto lo convierte en una herramienta eficaz de apoyo para la toma de decisiones en procesos de evaluación crediticia, especialmente en escenarios donde el costo del incumplimiento es elevado.


La matriz de confusión del modelo **Gradient Boosting** muestra un desempeño sólido en la identificación de clientes con riesgo de *default*. De un total de **1,517 clientes que realmente incurrieron en incumplimiento**, el modelo logró identificar correctamente **1,391 casos**, lo que refleja una **alta capacidad de detección del riesgo crediticio** (recall elevado para la clase *Default*).



### Comparación final de modelos de clasificación basados en algoritmos de ensamble: **Random Forest** y **Gradient Boosting**.
Ambos modelos mostraron un desempeño sólido; sin embargo, el modelo de **Gradient Boosting presentó el mejor desempeño global**.

Gradient Boosting alcanzó un **accuracy de 0.85**, un **recall de 0.92 para la clase Default** y un **AUC–ROC de 0.895**, superando ligeramente a Random Forest en las métricas más relevantes para la evaluación del riesgo crediticio. Estos resultados indican una mayor capacidad para identificar correctamente a los clientes con alta probabilidad de incumplimiento.

Si bien Random Forest mostró una alta precisión en la predicción de clientes riesgosos, Gradient Boosting logró detectar un mayor número de casos reales de default, lo que resulta crítico en escenarios donde el costo del incumplimiento es elevado.

En consecuencia, **Gradient Boosting se selecciona como el modelo más adecuado** para apoyar la toma de decisiones en procesos de evaluación crediticia, al ofrecer un mejor balance entre capacidad predictiva y control del riesgo.

